# Chat with Your PDF - RAG with Gemini

**What RAG means:** an LLM does not know what is inside *your* PDF. So instead of asking the LLM directly, we
1. **split** the PDF into small chunks,
2. turn every chunk into a list of numbers (an **embedding**) that captures its meaning,
3. when the user asks a question, embed the question too and **retrieve** the chunks whose embeddings are closest,
4. give only those chunks to the LLM and tell it: *answer only from this text, cite the page, and say "I don't know" if it is not there*.

**Pipeline of this notebook**

1. Setup (libraries, API key, settings)
2. Load the PDF (text + page numbers)
3. Chunking
4. Embeddings + vector index
5. Retrieval (find the relevant chunks)
6. Generation (LLM answers with citations)
7. "I don't know" handling (threshold calibration)
8. Evaluation (your own test questions)
9. Saving for the Streamlit app

Run the cells from top to bottom. The settings you may want to change are in Step 2.

## Part 1: Setup

In [1]:
## Step 1: Install the libraries 
%pip install google-genai pypdf numpy pandas

  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -----------------------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\Tanmay\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [29]:
## Step 2: Imports and project settings 
import os, re, time, getpass, json
from pathlib import Path

import numpy as np
import pandas as pd
from pypdf import PdfReader
from google import genai
from google.genai import types

# PDF 
PDF_PATH = Path(r"C:\Users\Tanmay\Desktop\Chat with pdf\Statistics notes.pdf")

# model names
GEN_MODEL = "gemini-3.5-flash"
EMBED_MODEL = "gemini-embedding-001"

# chunking
CHUNK_SIZE = 1000        # characters per chunk
CHUNK_OVERLAP = 200      # characters shared between neighbouring chunks (so sentences are not cut off)

# --- retrieval
TOP_K = 4                # how many chunks we give to the LLM
MIN_SIMILARITY = 0.30    # below this we say "I don't know" without calling the LLM (calibrated in Steps 12-13)

print("PDF exists:", PDF_PATH.exists())

PDF exists: True


In [30]:
## Step 3: Connect to Gemini 
API_KEY = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

# list the models your key can use (handy if a model name below does not work)
for m in client.models.list():
    name = m.name.replace("models/", "")
    if "gemini" in name and ("flash" in name or "embedding" in name):
        print(name)

Paste your Gemini API key:  ········


gemini-2.5-flash
gemini-2.5-flash-preview-tts
gemini-flash-latest
gemini-flash-lite-latest
gemini-2.5-flash-lite
gemini-2.5-flash-image
gemini-3-flash-preview
gemini-3.1-flash-lite-preview
gemini-3.1-flash-lite
gemini-3.1-flash-image-preview
gemini-3.1-flash-image
gemini-3.1-flash-lite-image
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-omni-flash-preview
gemini-omni-1.1-flash
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash
gemini-3.1-flash-tts-preview
gemini-3.8-flash-tts
gemini-3.8-flash-lite-tts
gemini-embedding-001
gemini-embedding-2-preview
gemini-embedding-2
gemini-2.5-flash-native-audio-latest
gemini-2.5-flash-native-audio-preview-09-2025
gemini-2.5-flash-native-audio-preview-12-2025
gemini-3.1-flash-live-preview


In [31]:
## Step 4: Find working models automatically and test them
print("Current settings -> GEN_MODEL:", GEN_MODEL, "| EMBED_MODEL:", EMBED_MODEL)

all_models = list(client.models.list())
clean = lambda m: m.name.replace("models/", "")
can = lambda m, action: action in (getattr(m, "supported_actions", None) or [])

gen_names = [clean(m) for m in all_models if can(m, "generateContent") and "gemini" in clean(m)]
emb_names = [clean(m) for m in all_models if can(m, "embedContent")]
print("Can generate text:", gen_names)
print("Can embed:", emb_names)

# find a generation model that really works
skip = ("image", "tts", "audio", "live", "robotics", "computer", "embedding")
preferred = ["gemini-3.8-flash", "gemini-3.5-flash", "gemini-3.5-flash-lite"]
pool = gen_names or [clean(m) for m in all_models]
candidates = [n for n in preferred if n in pool] + [n for n in pool if "flash" in n and not any(s in n for s in skip)]
candidates = list(dict.fromkeys(candidates))[:6]

for name in candidates:
    try:
        r = client.models.generate_content(model=name, contents="Reply with the single word: ready")
        GEN_MODEL = name
        print("Generation model OK:", name, "->", (r.text or "").strip())
        break
    except Exception as e:
        print("  not usable:", name, "|", str(e)[:100])
else:
    print("No generation model worked. Copy the lines above and send them to me.")

# find an embedding model that really works
for name in (emb_names or ["gemini-embedding-001"])[:4]:
    try:
        t = client.models.embed_content(model=name, contents=["hello"])
        EMBED_MODEL = name
        print("Embedding model OK:", name, "| vector length:", len(t.embeddings[0].values))
        break
    except Exception as e:
        print("  not usable:", name, "|", str(e)[:100])
else:
    print("No embedding model worked. Copy the lines above and send them to me.")

Current settings -> GEN_MODEL: gemini-3.5-flash | EMBED_MODEL: gemini-embedding-001
Can generate text: ['gemini-2.5-flash', 'gemini-2.5-pro', 'gemini-2.5-flash-preview-tts', 'gemini-2.5-pro-preview-tts', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-pro-latest', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-image', 'gemini-3-flash-preview', 'gemini-3.1-pro-preview', 'gemini-3.1-pro-preview-customtools', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3-pro-image-preview', 'gemini-3-pro-image', 'gemini-3.1-flash-image-preview', 'gemini-3.1-flash-image', 'gemini-3.1-flash-lite-image', 'gemini-nano-banana-2.1', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-omni-flash-preview', 'gemini-omni-1.1-flash', 'gemini-3.5-transcribe', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-4-argon', 'gemini-3.1-flash-tts-preview', 'gemini-3.8-flash-tts', 'gemini-3.8-flash-lite-tts', 'gemini-robotics-er-2-preview', 'gemini-2.5-computer-use-preview-10-2

## Part 2: Load the PDF

In [32]:
## Step 5: Read the PDF page by page (we keep the page number so answers can cite it)
reader = PdfReader(str(PDF_PATH))
pages = []
for i, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "")
    text = re.sub(r"[ \t]+", " ", text)          # collapse repeated spaces
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    pages.append({"page": i, "text": text})

total_chars = sum(len(p["text"]) for p in pages)
empty_pages = [p["page"] for p in pages if len(p["text"]) < 20]
print(f"Pages: {len(pages)} | Characters: {total_chars}")
if empty_pages:
    print("Pages with almost no text (scanned images?):", empty_pages[:20])
print("\nFirst 500 characters of page 1:\n", pages[0]["text"][:500])

Pages: 38 | Characters: 47790

First 500 characters of page 1:
 STATISTICS FOR DATA ANALYTICS 
OVERVIEW OF STATISTICS 
Introduction to Statistics 
Descriptive And Inferential Statistics 
Basic Terms Of Statistics 
Types Of Data 
HARNESSING DATA 
Random Sampling 
Sampling With Replacement and Without 
Replacement 
Cochran's Minimum Sample Size 
Types of Sampling 
Simple Random Sampling 
Stratified Random Sampling 
Cluster Random Sampling 
Systematic Random Sampling 
Multi stage Sampling 
Sampling Error 
Methods Of Collecting Data 
EXPLORATORY DATA ANALYSIS 
E


## Part 3: Chunking

An LLM prompt cannot hold a whole book, and a search over whole pages is too blunt. So we cut the text into overlapping pieces of about 1,000 characters.
Each chunk remembers which page it came from.

In [33]:
## Step 6: Split the pages into overlapping chunks
def chunk_pages(pages, size=1000, overlap=200):
    chunks = []
    for p in pages:
        text = p["text"]
        if len(text) < 20:
            continue
        start = 0
        while start < len(text):
            end = min(start + size, len(text))
            # try to end at a sentence boundary so chunks read naturally
            if end < len(text):
                cut = max(text.rfind(". ", start, end), text.rfind("\n", start, end))
                if cut > start + size // 2:
                    end = cut + 1
            piece = text[start:end].strip()
            if piece:
                chunks.append({"id": len(chunks), "page": p["page"], "text": piece})
            if end >= len(text):
                break
            start = max(end - overlap, start + 1)
    return chunks

chunks = chunk_pages(pages, CHUNK_SIZE, CHUNK_OVERLAP)
lengths = [len(c["text"]) for c in chunks]
print(f"{len(chunks)} chunks | average length {np.mean(lengths):.0f} characters | shortest {min(lengths)} | longest {max(lengths)}")
print("\nExample chunk (page %d):\n%s" % (chunks[0]["page"], chunks[0]["text"][:400]))

71 chunks | average length 765 characters | shortest 210 | longest 998

Example chunk (page 1):
STATISTICS FOR DATA ANALYTICS 
OVERVIEW OF STATISTICS 
Introduction to Statistics 
Descriptive And Inferential Statistics 
Basic Terms Of Statistics 
Types Of Data 
HARNESSING DATA 
Random Sampling 
Sampling With Replacement and Without 
Replacement 
Cochran's Minimum Sample Size 
Types of Sampling 
Simple Random Sampling 
Stratified Random Sampling 
Cluster Random Sampling 
Systematic Random Samp


## Part 4: Embeddings and the vector index

An embedding turns text into a list of numbers so that texts with similar meaning end up close together.
Our "vector store" is simply a NumPy matrix with one row per chunk. For a single PDF this is all we need, and it keeps RAM use tiny.

In [34]:
## Step 7: Helper to embed text with automatic retry (the free API tier has rate limits)
def embed_texts(texts, task_type, batch_size=50, pause=1.0):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        for attempt in range(6):
            try:
                resp = client.models.embed_content(
                    model=EMBED_MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(task_type=task_type),
                )
                vectors.extend([e.values for e in resp.embeddings])
                break
            except Exception as e:
                wait = 2 ** attempt * 2
                print(f"  API error ({str(e)[:80]}...). Retrying in {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError("Embedding failed after several retries")
        time.sleep(pause)
    arr = np.array(vectors, dtype=np.float32)
    arr /= np.linalg.norm(arr, axis=1, keepdims=True)    # normalise so a dot product = cosine similarity
    return arr

In [35]:
## Step 8: Embed every chunk 
cache_file = Path(f"index_{PDF_PATH.stem}_{CHUNK_SIZE}_{CHUNK_OVERLAP}.npz")

if cache_file.exists():
    chunk_vectors = np.load(cache_file)["vectors"]
    print("Loaded embeddings from cache:", chunk_vectors.shape)
else:
    start = time.time()
    chunk_vectors = embed_texts([c["text"] for c in chunks], task_type="RETRIEVAL_DOCUMENT")
    np.savez_compressed(cache_file, vectors=chunk_vectors)
    print(f"Embedded {len(chunks)} chunks in {time.time() - start:.0f}s -> matrix {chunk_vectors.shape}")

Loaded embeddings from cache: (71, 3072)


## Part 5: Retrieval

In [36]:
## Step 9: Retrieval function - find the chunks closest in meaning to a question
def retrieve(question, top_k=TOP_K):
    q_vec = embed_texts([question], task_type="RETRIEVAL_QUERY", pause=0)[0]
    scores = chunk_vectors @ q_vec                      # cosine similarity with every chunk
    best = np.argsort(scores)[::-1][:top_k]
    return [{**chunks[i], "score": float(scores[i])} for i in best]

for r in retrieve("What is this document about?"):
    print(f"page {r['page']:>3} | score {r['score']:.3f} | {r['text'][:120]!r}")

page  23 | score 0.641 | 'sitive covariance → Variables move together. \n• Negative covariance → Variables move oppositely. \n• Zero covariance → No'
page   1 | score 0.640 | 'STATISTICS FOR DATA ANALYTICS \nOVERVIEW OF STATISTICS \nIntroduction to Statistics \nDescriptive And Inferential Statistic'
page   7 | score 0.640 | 'llecting Data in Research \nData collection is a crucial step in research, business, and decision-making. The choice of m'
page  13 | score 0.635 | 'way to understand the spread of data in a normal distribution. \n \nImportance and Applications \n• Statistical Inference: '


## Part 6: Generation

The prompt is the most important design choice here. We tell the model to use only the supplied context, to cite pages,
and to admit when the answer is missing. That instruction is what makes a RAG bot trustworthy.

In [37]:
## Step 10: Build the prompt and ask Gemini
SYSTEM_INSTRUCTIONS = (
    "You are a careful assistant that answers questions about one PDF document. "
    "Use ONLY the context excerpts provided. Do not use outside knowledge. "
    "Cite the page for each fact like (p. 3). "
    "If the context does not contain the answer, reply exactly: "
    "\"I couldn't find that in the document.\" "
    "Keep answers clear and concise."
)

def build_prompt(question, retrieved):
    context = "\n\n".join(f"[Excerpt {i + 1} | page {r['page']}]\n{r['text']}" for i, r in enumerate(retrieved))
    return f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"

def generate_answer(question, retrieved):
    for attempt in range(5):
        try:
            resp = client.models.generate_content(
                model=GEN_MODEL,
                contents=build_prompt(question, retrieved),
                config=types.GenerateContentConfig(system_instruction=SYSTEM_INSTRUCTIONS, temperature=0.2),
            )
            return (resp.text or "").strip()
        except Exception as e:
            wait = 2 ** attempt * 2
            print(f"  API error ({str(e)[:80]}...). Retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError("Generation failed after several retries")

In [22]:
## Step 11: The full RAG function - retrieve, check relevance, generate
NOT_FOUND = "I couldn't find that in the document."

def ask(question, top_k=TOP_K, min_similarity=None, verbose=True):
    min_similarity = MIN_SIMILARITY if min_similarity is None else min_similarity
    retrieved = retrieve(question, top_k)
    if retrieved[0]["score"] < min_similarity:                 # nothing relevant -> do not even call the LLM
        answer = NOT_FOUND
    else:
        answer = generate_answer(question, retrieved)
    if verbose:
        print("Q:", question)
        print("A:", answer)
        print("Sources:", sorted({r["page"] for r in retrieved}), "| best score: %.3f" % retrieved[0]["score"])
    return {"question": question, "answer": answer, "sources": retrieved}

_ = ask("What is this document about?")

Q: What is this document about?
A: Based on the provided context, the document is about **Statistics for Data Analytics** (p. 1). It covers fundamental statistical concepts and topics including:

* **Overview of Statistics:** Descriptive and inferential statistics, basic terms, and types of data (p. 1).
* **Harnessing Data:** Sampling methods, sample size, sampling errors, and primary/secondary data collection methods like surveys and questionnaires (p. 1, p. 7).
* **Exploratory Data Analysis:** Measures of central tendency and spread, data distributions (normal distribution, empirical rule), skewness, kurtosis, distance measures, and covariance and correlation (p. 1, p. 13, p. 23).
* **Hypothesis Testing:** Using statistical methods and sample data to make population decisions and determine statistical significance (p. 1, p. 23).
Sources: [1, 7, 13, 23] | best score: 0.641


## Part 7: Knowing what it does not know

A good RAG bot must refuse when the document does not contain the answer. We use two safety nets:
the prompt instruction (the LLM says "I couldn't find that") and a similarity threshold (we skip the LLM when nothing is close).
Next we pick the threshold using real numbers instead of guessing.

In [38]:
## Step 12: Calibrate the "I don't know" threshold
# EDIT these two lists: 4-5 questions the PDF CAN answer, and 4-5 it definitely CANNOT.
in_doc_questions = [
    "What is stratified random sampling?",
    "What does the empirical rule say about a normal distribution?",
    "What is the difference between positive and negative covariance?",
    "How do you use the critical region in decision-making?",
    "What is Cochran's minimum sample size?",
    "What are the types of sampling?",
]
out_of_doc_questions = [
    "Who won the football world cup in 2018?",
    "How do I bake a chocolate cake?",
    "What is a neural network?",
    "How does a random forest work?",
    "What is the capital of Australia?",
]

def best_score(q):
    return retrieve(q, 1)[0]["score"]

in_scores = [best_score(q) for q in in_doc_questions]
out_scores = [best_score(q) for q in out_of_doc_questions]
print("Questions the PDF can answer -> best scores:", [round(s, 3) for s in in_scores])
print("Questions it cannot answer   -> best scores:", [round(s, 3) for s in out_scores])
print("\nPick a MIN_SIMILARITY between the highest 'cannot' score and the lowest 'can' score.")
print("Suggestion:", round((max(out_scores) + min(in_scores)) / 2, 3))

Questions the PDF can answer -> best scores: [0.723, 0.762, 0.77, 0.802, 0.779, 0.767]
Questions it cannot answer   -> best scores: [0.526, 0.551, 0.589, 0.618, 0.546]

Pick a MIN_SIMILARITY between the highest 'cannot' score and the lowest 'can' score.
Suggestion: 0.67


In [39]:
## Step 13: Apply the threshold and test refusing
MIN_SIMILARITY = round((max(out_scores) + min(in_scores)) / 2, 3)
print("MIN_SIMILARITY set to", MIN_SIMILARITY)

_ = ask("Who won the football world cup in 2018?")

MIN_SIMILARITY set to 0.67
Q: Who won the football world cup in 2018?
A: I couldn't find that in the document.
Sources: [7, 8, 9, 33] | best score: 0.526


## Part 8: Evaluation

Do not just trust one demo question. Write ~8 questions yourself, record the expected answer from reading the PDF, run the bot, and judge each result.
This table is also great material for your submission report.

In [40]:
## Step 14: Run your own test questions (EDIT the list: use real questions about YOUR PDF)
test_set = [
    {"question": "What is stratified random sampling?", "expected": "<from your notes>", "answerable": True},
{"question": "What does the empirical rule say about a normal distribution?", "expected": "<from your notes>", "answerable": True},
{"question": "What is the difference between positive and negative covariance?", "expected": "<from your notes>", "answerable": True},
{"question": "How do you use the critical region in decision-making?", "expected": "<from your notes>", "answerable": True},
{"question": "What is Cochran's minimum sample size?", "expected": "<from your notes>", "answerable": True},
{"question": "What is the capital of Australia?", "expected": "Should refuse", "answerable": False},
    {"question": "How does a random forest work?", "expected": "Should refuse (check the PDF doesn't cover it)", "answerable": False},
]

rows = []
for t in test_set:
    out = ask(t["question"], verbose=False)
    refused = out["answer"].startswith("I couldn't find")
    rows.append({
        "question": t["question"],
        "expected": t["expected"],
        "bot_answer": out["answer"],
        "pages_retrieved": sorted({s["page"] for s in out["sources"]}),
        "best_score": round(out["sources"][0]["score"], 3),
        "refused": refused,
        "behaved_correctly_on_refusal": (refused == (not t["answerable"])),
    })
    time.sleep(1)

results_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
results_df

  API error (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp...). Retrying in 2s


,question,expected,bot_answer,pages_retrieved,best_score,refused,behaved_correctly_on_refusal
0,What is stratified random sampling?,<from your notes>,Stratified Sampling is a method where:\n* The population is divided into subgroups (strata) based on shared characte...,"[1, 2, 3]",0.723,False,True
1,What does the empirical rule say about a normal distribution?,<from your notes>,The Empirical Rule states that for normally distributed (bell-shaped) data:\n* **68%** of data falls within ±1 stand...,"[13, 14]",0.762,False,True
2,What is the difference between positive and negative covariance?,<from your notes>,Based on the provided document:\n\n* **Positive covariance** means the variables tend to move together in the same d...,"[21, 22, 23]",0.770,False,True
3,How do you use the critical region in decision-making?,<from your notes>,"To use the critical region in decision-making:\n\n1. **Compute the test statistic** (e.g., z-score, t-score) (p. 26)...","[25, 26, 27]",0.802,False,True
4,What is Cochran's minimum sample size?,<from your notes>,"Based on the provided document, Cochran's minimum sample size ($n_0$) refers to the minimum required sample size cal...","[5, 6, 7]",0.779,False,True
5,What is the capital of Australia?,Should refuse,I couldn't find that in the document.,"[7, 8, 9]",0.546,True,True
6,How does a random forest work?,Should refuse (check the PDF doesn't cover it),I couldn't find that in the document.,"[1, 2, 4, 23]",0.618,True,True


In [41]:
for i, row in results_df.iterrows():
    print(f"[{i}] Q: {row['question']}\nA: {row['bot_answer']}\nPages: {row['pages_retrieved']}\n" + "-" * 70)

[0] Q: What is stratified random sampling?
A: Stratified Sampling is a method where:
* The population is divided into subgroups (strata) based on shared characteristics such as age or gender (p. 2).
* Samples are then randomly selected from each stratum (p. 2).
* It ensures the representation of all key groups (p. 2).
Pages: [1, 2, 3]
----------------------------------------------------------------------
[1] Q: What does the empirical rule say about a normal distribution?
A: The Empirical Rule states that for normally distributed (bell-shaped) data:
* **68%** of data falls within ±1 standard deviation ($\sigma$) of the mean ($\mu$) (p. 14).
* **95%** of data falls within ±2$\sigma$ of the mean (p. 14).
* **99.7%** of data falls within ±3$\sigma$ of the mean (p. 14).

It provides a quick way to understand the spread of data in a normal distribution (p. 13).
Pages: [13, 14]
----------------------------------------------------------------------
[2] Q: What is the difference between positi

In [43]:
## Step 15: Score the evaluation
results_df["answerable"] = [t["answerable"] for t in test_set]

# AFTER reading every bot_answer yourself, fill this list in the same order as test_set.
# For answerable questions: True = the answer is correct, False = wrong or wrongly refused.
# For the "should refuse" questions: True = it refused, False = it answered anyway.
manual_correct = [True, True, True, True, True, True, True]   

if None in manual_correct:
    print("Read the bot_answer column above, fill manual_correct with True/False, then re-run this cell.")
else:
    results_df["correct"] = manual_correct
    ans = results_df[results_df["answerable"]]
    ref = results_df[~results_df["answerable"]]
    print(f"Answerable questions: {int(ans['correct'].sum())} of {len(ans)} correct")
    print(f"Should-refuse questions: {int(ref['correct'].sum())} of {len(ref)} correctly refused")
    print(f"Overall: {int(results_df['correct'].sum())} of {len(results_df)}")

Answerable questions: 5 of 5 correct
Should-refuse questions: 2 of 2 correctly refused
Overall: 7 of 7


## Part 9: Save for the Streamlit app

In [44]:
## Step 16: Save the index (chunks + vectors) so the app can start instantly
with open("rag_chunks.json", "w", encoding="utf-8") as f:
    json.dump({"pdf": PDF_PATH.name, "chunks": chunks, "min_similarity": MIN_SIMILARITY,
               "gen_model": GEN_MODEL, "embed_model": EMBED_MODEL}, f)
np.save("rag_vectors.npy", chunk_vectors)
print("Saved rag_chunks.json and rag_vectors.npy")
print("The Streamlit app (app_rag.py) can also build its own index from any uploaded PDF.")

Saved rag_chunks.json and rag_vectors.npy
The Streamlit app (app_rag.py) can also build its own index from any uploaded PDF.


In [45]:
## Step 17: Summary for your report
print("Document:", PDF_PATH.name, "|", len(pages), "pages |", len(chunks), "chunks")
print("Chunking:", CHUNK_SIZE, "chars with", CHUNK_OVERLAP, "overlap | top-k:", TOP_K)
print("Embedding model:", EMBED_MODEL, "| Generation model:", GEN_MODEL)
print("Refusal threshold (cosine similarity):", MIN_SIMILARITY)

Document: Statistics notes.pdf | 38 pages | 71 chunks
Chunking: 1000 chars with 200 overlap | top-k: 4
Embedding model: gemini-embedding-001 | Generation model: gemini-3.5-flash
Refusal threshold (cosine similarity): 0.67
